# Day 1 · Section 7: Transformer Self-Attention

Standalone student notebook. Run cells in order, change the examples, and use the checks to explain what happened. It contains no workshop slides. CPU exercises work without downloads; optional Qwen3 cells require network access and, where indicated, a suitable GPU.


## Goals · 7.1–7.7

Build Q, K and V projections, compute all query–key similarities, scale and normalize them, retrieve weighted Values, split an output into multiple heads and add a residual/normalization/MLP toy block. Causal masking is isolated in Section 8.


In [ ]:
import sys, subprocess, numpy as np
try:
    import torch
    DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print('PyTorch',torch.__version__,'device',DEVICE)
except ImportError:
    torch=None;DEVICE=None
    print('PyTorch unavailable locally; NumPy path remains runnable. Colab normally includes PyTorch.')
rng=np.random.default_rng(7)


### 7.2 · Learned projections and shapes

The example matrices stand in for learned weights. `[sequence, hidden] @ [hidden, head_dim]` produces one Q, K or V vector per position.


In [ ]:
X=np.array([[1.,0.,.5],[0.,1.,.5],[.5,.5,1.]])
Wq=np.array([[1.,0.],[0.,1.],[.5,.5]])
Wk=np.array([[.8,.2],[.2,.8],[.5,.5]])
Wv=np.array([[1.,0.],[0.,1.],[1.,1.]])
Q,K,V=X@Wq,X@Wk,X@Wv
print('X',X.shape,'Q/K/V',Q.shape,K.shape,V.shape)
assert Q.shape==K.shape==V.shape==(3,2)
if torch is not None:
    Xt=torch.tensor(X,dtype=torch.float32,device=DEVICE)
    Qt=Xt@torch.tensor(Wq,dtype=torch.float32,device=DEVICE)
    Kt=Xt@torch.tensor(Wk,dtype=torch.float32,device=DEVICE)
    Vt=Xt@torch.tensor(Wv,dtype=torch.float32,device=DEVICE)
    assert np.allclose(Qt.cpu().numpy(),Q,atol=1e-6)


### 7.3–7.5 · Scores → weights → weighted Values

Rows of `Q @ K.T` correspond to Queries and columns to Keys. Divide by √head_dim before softmax to reduce extreme score magnitudes. Normalize over the Key axis for each Query.


In [ ]:
def softmax_np(z,axis=-1):
    z=z-np.max(z,axis=axis,keepdims=True);e=np.exp(z);return e/e.sum(axis=axis,keepdims=True)
raw_scores=Q@K.T
scaled=raw_scores/np.sqrt(K.shape[-1])
weights=softmax_np(scaled,axis=-1)
context=weights@V
print('raw scores:\n',np.round(raw_scores,2))
print('weights:\n',np.round(weights,3),'row sums:',weights.sum(axis=-1))
print('weighted Values:\n',np.round(context,3))
assert raw_scores.shape==(3,3) and context.shape==(3,2)
if torch is not None:
    torch_weights=torch.softmax((Qt@Kt.transpose(-2,-1))/np.sqrt(Kt.shape[-1]),dim=-1)
    torch_context=torch_weights@Vt
    assert np.allclose(torch_context.cpu().numpy(),context,atol=1e-6)


### 7.6 · Multi-head intuition

Use two distinct projection sets, concatenate their outputs and project back to hidden width. In a real model head weights are learned; different heads may attend to different relationships.


In [ ]:
def head_output(x,wq,wk,wv):
    q,k,v=x@wq,x@wk,x@wv
    return softmax_np(q@k.T/np.sqrt(q.shape[-1]))@v
head1=head_output(X,Wq,Wk,Wv)
head2=head_output(X,Wk,Wq,Wv/2)
joined=np.concatenate([head1,head2],axis=-1)
Wo=np.array([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.],[.2,.2,.2]])
multi_head=joined@Wo
print('head1',head1.shape,'head2',head2.shape,'joined',joined.shape,'projected',multi_head.shape)
assert multi_head.shape==X.shape


### 7.7 · A toy Transformer block

Residual connections add the input back. Normalization changes feature scale; a feed-forward transformation acts independently at each token position. This is a structural sketch, not a Qwen layer implementation.


In [ ]:
def layer_norm_np(x,eps=1e-5):
    return (x-x.mean(axis=-1,keepdims=True))/np.sqrt(x.var(axis=-1,keepdims=True)+eps)
after_attention=layer_norm_np(X+multi_head)
W_up=np.eye(3)*.5;W_down=np.eye(3)
ffn=np.maximum(after_attention@W_up,0)@W_down
block_output=layer_norm_np(after_attention+ffn)
print('input',X.shape,'after attention',after_attention.shape,'after block',block_output.shape)
assert block_output.shape==X.shape
if torch is not None:
    torch_normalized=torch.nn.functional.layer_norm(torch.tensor(X+multi_head,dtype=torch.float32,device=DEVICE),(3,))
    assert np.allclose(torch_normalized.cpu().numpy(),after_attention,atol=1e-4)


### Inspect model configuration without weights (optional download)

Qwen3-4B has different numbers of Query and Key/Value heads. The projection output width is reshaped into heads × head width; the sequence axis is not split into heads.


In [ ]:
import sys, subprocess
IN_COLAB='google.colab' in sys.modules
RUN_DOWNLOADS=IN_COLAB  # set True locally when network/package installation is intended
print('Download optional model assets:',RUN_DOWNLOADS)


In [ ]:
if RUN_DOWNLOADS:
    subprocess.check_call([sys.executable,'-m','pip','-q','install','transformers>=4.52.4,<6'])
    from transformers import AutoConfig
    config=AutoConfig.from_pretrained('Qwen/Qwen3-4B')
    print('hidden',config.hidden_size,'Q heads',config.num_attention_heads,
          'KV heads',config.num_key_value_heads,'head width',config.head_dim)
    assert config.num_attention_heads>=config.num_key_value_heads
else:print('Configuration download skipped.')


## Checks

1. Which axis of the score matrix corresponds to Keys? Why normalize it?
2. What changes if you remove scaling? Try multiplying all Q values by 10.
3. Why does the concatenated multi-head output need an output projection?
4. Which operations mix tokens, and which operate on each token separately?
